# Notebook 07 — Combined Voltage and Geometry Optimization

This notebook combines the earlier voltage and geometry sweeps into a single reduced design workflow.

It focuses on:

- varying both voltages and lower-boundary electrode geometry
- solving Laplace's equation for each candidate design
- computing electric fields and normalized pseudopotentials
- locating a candidate interior trap point for each design
- extracting Hessian-based curvature diagnostics
- scoring each design with a simple curvature-versus-height objective
- identifying a best global configuration in the scanned parameter space

This is still a lightweight model, but it gives a first combined design-space search for a surface-style ion-trap layout.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Grid and fixed geometric conventions

We keep a fixed computational grid and symmetric lower-boundary layout conventions. The design variables are:

- center RF-like voltage
- side DC-like voltage
- center electrode width
- gap between the center and side electrodes

The side-electrode width is kept fixed in this reduced combined scan.


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Fixed side-electrode width
side_width = 1.3

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')
print(f'Fixed side-electrode width = {side_width:.2f}')


## Helper functions


In [ ]:
def idx(i, j, nx):
    return i * nx + j

def make_electrode_masks(center_width, gap, side_width):
    center_half = center_width / 2.0
    left_center = -(center_half + gap + side_width / 2.0)
    right_center = +(center_half + gap + side_width / 2.0)

    left_mask = (x >= left_center - side_width / 2.0) & (x <= left_center + side_width / 2.0)
    center_mask = (x >= -center_half) & (x <= center_half)
    right_mask = (x >= right_center - side_width / 2.0) & (x <= right_center + side_width / 2.0)
    remaining_ground = ~(left_mask | center_mask | right_mask)
    return left_mask, center_mask, right_mask, remaining_ground

def build_boundary_problem(left_mask, center_mask, right_mask, remaining_ground, v_rf, v_dc):
    V = np.zeros((ny, nx), dtype=float)
    fixed = np.zeros((ny, nx), dtype=bool)

    # Ground outer boundaries
    V[:, 0] = 0.0
    V[:, -1] = 0.0
    V[-1, :] = 0.0
    fixed[:, 0] = True
    fixed[:, -1] = True
    fixed[-1, :] = True

    # Lower boundary electrodes
    bottom = 0
    V[bottom, left_mask] = v_dc
    V[bottom, center_mask] = v_rf
    V[bottom, right_mask] = v_dc
    V[bottom, remaining_ground] = 0.0
    fixed[bottom, :] = True
    return V, fixed

def solve_laplace(V, fixed):
    N = nx * ny
    A = lil_matrix((N, N), dtype=float)
    b = np.zeros(N, dtype=float)

    cx = 1.0 / dx**2
    cy = 1.0 / dy**2
    c0 = -2.0 * (cx + cy)

    for i in range(ny):
        for j in range(nx):
            k = idx(i, j, nx)

            if fixed[i, j]:
                A[k, k] = 1.0
                b[k] = V[i, j]
            else:
                A[k, idx(i, j, nx)] = c0
                A[k, idx(i, j - 1, nx)] = cx
                A[k, idx(i, j + 1, nx)] = cx
                A[k, idx(i - 1, j, nx)] = cy
                A[k, idx(i + 1, j, nx)] = cy

    sol = spsolve(A.tocsr(), b)
    return sol.reshape((ny, nx))

def compute_pseudopotential(Vsol):
    dV_dy, dV_dx = np.gradient(Vsol, dy, dx)
    Ex = -dV_dx
    Ey = -dV_dy
    E_mag = np.sqrt(Ex**2 + Ey**2)
    Phi_pseudo = E_mag**2
    return Ex, Ey, E_mag, Phi_pseudo

def find_candidate_point(Phi_pseudo):
    margin_i_low = 2
    margin_i_high = 3
    margin_j = 3
    i_slice = slice(margin_i_low, ny - margin_i_high)
    j_slice = slice(margin_j, nx - margin_j)
    Phi_search = Phi_pseudo[i_slice, j_slice]
    min_flat = np.argmin(Phi_search)
    min_i_local, min_j_local = np.unravel_index(min_flat, Phi_search.shape)
    min_i = min_i_local + margin_i_low
    min_j = min_j_local + margin_j
    return min_i, min_j

def second_derivatives(F, i, j):
    d2F_dx2 = (F[i, j + 1] - 2.0 * F[i, j] + F[i, j - 1]) / dx**2
    d2F_dy2 = (F[i + 1, j] - 2.0 * F[i, j] + F[i - 1, j]) / dy**2
    d2F_dxdy = (
        F[i + 1, j + 1] - F[i + 1, j - 1] - F[i - 1, j + 1] + F[i - 1, j - 1]
    ) / (4.0 * dx * dy)
    return d2F_dx2, d2F_dy2, d2F_dxdy

def design_metrics(Phi_pseudo):
    i, j = find_candidate_point(Phi_pseudo)
    trap_x = x[j]
    trap_y = y[i]
    trap_value = Phi_pseudo[i, j]

    center = Phi_pseudo[i, j]
    neighbors = np.array([
        Phi_pseudo[i - 1, j],
        Phi_pseudo[i + 1, j],
        Phi_pseudo[i, j - 1],
        Phi_pseudo[i, j + 1],
    ])
    is_local_min = np.all(center <= neighbors)

    d2x, d2y, d2xy = second_derivatives(Phi_pseudo, i, j)
    H = np.array([[d2x, d2xy], [d2xy, d2y]], dtype=float)
    eigvals, eigvecs = np.linalg.eigh(H)

    min_curvature = float(np.min(eigvals))
    positive_sum = float(np.sum(np.clip(eigvals, 0.0, None)))
    omega_proxy = np.sqrt(np.clip(eigvals, 0.0, None))

    # Require both local-minimum behavior and positive principal curvatures
    valid_trap = bool(is_local_min and (min_curvature > 0.0))

    if valid_trap:
        trap_strength = positive_sum
        # Objective: stronger curvature and lower height are preferred
        objective_score = trap_strength / max(trap_y, 1e-6)
    else:
        trap_strength = 0.0
        objective_score = 0.0

    return {
        'i': i,
        'j': j,
        'trap_x': trap_x,
        'trap_y': trap_y,
        'trap_value': trap_value,
        'is_local_min': bool(is_local_min),
        'valid_trap': valid_trap,
        'H': H,
        'eigvals': eigvals,
        'omega_proxy': omega_proxy,
        'trap_strength': trap_strength,
        'min_curvature': min_curvature,
        'objective_score': objective_score,
    }


## Combined parameter grid

We scan a compact design space:

- `v_rf`: center RF-like voltage
- `v_dc`: side DC-like voltage
- `center_width`: center-electrode width
- `gap`: spacing between the center and side electrodes

For visualization, we aggregate the best score over the voltage subspace for each geometry pair.


In [ ]:
v_rf_values = np.array([0.8, 1.2, 1.6])
v_dc_values = np.array([-2.0, -1.5, -1.0])
center_width_values = np.array([0.6, 0.8, 1.0, 1.2])
gap_values = np.array([0.2, 0.35, 0.5, 0.65])

best_score_map = np.zeros((len(gap_values), len(center_width_values)))
best_height_map = np.zeros((len(gap_values), len(center_width_values)))
best_strength_map = np.zeros((len(gap_values), len(center_width_values)))
valid_count_map = np.zeros((len(gap_values), len(center_width_values)))

all_results = []
best_by_geometry = {}

total_cases = len(v_rf_values) * len(v_dc_values) * len(center_width_values) * len(gap_values)
print(f'Scanning {total_cases} combined designs...')


## Run the combined scan


In [ ]:
for igap, gap in enumerate(gap_values):
    for iw, center_width in enumerate(center_width_values):
        local_best_score = 0.0
        local_best_metrics = None
        local_valid_count = 0

        left_mask, center_mask, right_mask, remaining_ground = make_electrode_masks(
            center_width=center_width,
            gap=gap,
            side_width=side_width,
        )

        for v_dc in v_dc_values:
            for v_rf in v_rf_values:
                V, fixed = build_boundary_problem(
                    left_mask=left_mask,
                    center_mask=center_mask,
                    right_mask=right_mask,
                    remaining_ground=remaining_ground,
                    v_rf=v_rf,
                    v_dc=v_dc,
                )
                Vsol = solve_laplace(V, fixed)
                Ex, Ey, E_mag, Phi_pseudo = compute_pseudopotential(Vsol)
                metrics = design_metrics(Phi_pseudo)

                record = {
                    'gap': float(gap),
                    'center_width': float(center_width),
                    'v_rf': float(v_rf),
                    'v_dc': float(v_dc),
                    'left_mask': left_mask,
                    'center_mask': center_mask,
                    'right_mask': right_mask,
                    'remaining_ground': remaining_ground,
                    'Vsol': Vsol,
                    'Phi_pseudo': Phi_pseudo,
                    **metrics,
                }
                all_results.append(record)

                if metrics['valid_trap']:
                    local_valid_count += 1

                if metrics['objective_score'] > local_best_score:
                    local_best_score = metrics['objective_score']
                    local_best_metrics = record

        valid_count_map[igap, iw] = local_valid_count

        if local_best_metrics is not None:
            best_score_map[igap, iw] = local_best_metrics['objective_score']
            best_height_map[igap, iw] = local_best_metrics['trap_y']
            best_strength_map[igap, iw] = local_best_metrics['trap_strength']
            best_by_geometry[(float(gap), float(center_width))] = local_best_metrics
        else:
            best_by_geometry[(float(gap), float(center_width))] = None

print('Combined scan complete.')


## Best objective score by geometry

For each geometry pair, this map shows the best score obtained over the scanned voltage pairs.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    best_score_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
)
ax.set_title('Best Objective Score vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Best score = trap_strength / trap_height')
plt.tight_layout()
plt.show()


## Best trap-height map by geometry


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    best_height_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
)
ax.set_title('Best Trap Height vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Best trap height over scanned voltages')
plt.tight_layout()
plt.show()


## Best trap-strength map by geometry


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    best_strength_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
)
ax.set_title('Best Trap Strength vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Best trap strength over scanned voltages')
plt.tight_layout()
plt.show()


## Valid-trap count by geometry


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(
    valid_count_map,
    origin='lower',
    aspect='auto',
    extent=[center_width_values[0], center_width_values[-1], gap_values[0], gap_values[-1]],
    vmin=0,
    vmax=len(v_rf_values) * len(v_dc_values),
)
ax.set_title('Valid-Trap Count vs Geometry Pair')
ax.set_xlabel('Center electrode width')
ax.set_ylabel('Gap')
fig.colorbar(im, ax=ax, label='Number of valid voltage pairs')
plt.tight_layout()
plt.show()


## Best global design

We identify the single best design across the full scanned parameter space using the objective score.


In [ ]:
global_best = max(all_results, key=lambda r: r['objective_score'])

print('Best global design:')
print(f"  gap = {global_best['gap']:.3f}")
print(f"  center_width = {global_best['center_width']:.3f}")
print(f"  v_rf = {global_best['v_rf']:.3f}")
print(f"  v_dc = {global_best['v_dc']:.3f}")
print(f"  trap_x = {global_best['trap_x']:.3f}")
print(f"  trap_y = {global_best['trap_y']:.3f}")
print(f"  valid_trap = {global_best['valid_trap']}")
print(f"  trap_strength = {global_best['trap_strength']:.3e}")
print(f"  min_curvature = {global_best['min_curvature']:.3e}")
print(f"  objective_score = {global_best['objective_score']:.3e}")
print('  curvature eigenvalues:')
print(global_best['eigvals'])
print('  secular-frequency proxies:')
print(global_best['omega_proxy'])


## Visualize the best global pseudopotential configuration


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.pcolormesh(X, Y, global_best['Phi_pseudo'], shading='auto')
cs = ax.contour(X, Y, global_best['Phi_pseudo'], levels=15)
ax.clabel(cs, inline=True, fontsize=8)
ax.scatter([global_best['trap_x']], [global_best['trap_y']], s=90, label='Best global trap point')
ax.plot(x[global_best['left_mask']], np.full(np.sum(global_best['left_mask']), y[0]), linewidth=7)
ax.plot(x[global_best['center_mask']], np.full(np.sum(global_best['center_mask']), y[0]), linewidth=7)
ax.plot(x[global_best['right_mask']], np.full(np.sum(global_best['right_mask']), y[0]), linewidth=7)
ax.set_title('Best Global Design: Normalized RF-Style Pseudopotential')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
fig.colorbar(im, ax=ax, label='Pseudo')
plt.tight_layout()
plt.show()


## Best global lower-boundary geometry


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))

ax.fill_between(x[global_best['left_mask']], -0.12, 0.12, label='Left DC-like')
ax.fill_between(x[global_best['center_mask']], -0.12, 0.12, label='Center RF-like')
ax.fill_between(x[global_best['right_mask']], -0.12, 0.12, label='Right DC-like')

ax.set_xlabel('x')
ax.set_ylim(-0.35, 0.45)
ax.set_yticks([])
ax.set_aspect('auto')
ax.grid(False)
ax.legend(loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.28), frameon=True)
ax.set_title('Best Global Design: Lower-Boundary Electrode Layout', pad=28)

plt.tight_layout()
plt.show()


## Notes and next steps

This notebook provides a first combined voltage-plus-geometry design scan for the reduced surface-style trap model.

Natural next steps:

- refine the scan around the best region with a denser grid
- add explicit physical constants for SI-scale frequency estimates
- include axial confinement and DC compensation structure
- compare alternative objective functions
- move from grid search to a true optimization algorithm
- benchmark runtime and refactor reusable solver components into `src/`

That progression will move the workflow toward a more practical trapped-ion design toolkit.
